# MedGemma-4B: data prep + LoRA fine-tune (Kaggle T4)
Settings: Accelerator GPU T4 x2, Internet ON. Secrets: `HF_TOKEN` (read) and optional `HF_WRITE_TOKEN` (write, used only to upload results to a PRIVATE HF repo).
Mitigation in this run: LoRA on VQA-RAD with answer-only loss and yes/no balancing (reduces false-"yes" bias). Abstention is handled at inference time on the laptop.
Run cells top to bottom. Cell 3 builds the splits, cell 4 uploads them so your teammate can start, cells 5-7 train.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate pillow datasets

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    token = _s.get_secret("HF_TOKEN")
    try: wtoken = _s.get_secret("HF_WRITE_TOKEN")
    except Exception: wtoken = None
except Exception:
    token, wtoken = os.environ.get("HF_TOKEN"), os.environ.get("HF_WRITE_TOKEN")
assert token, "Add HF_TOKEN as a Kaggle secret"
from huggingface_hub import login
login(token=token)
print("logged in | write token present:", bool(wtoken))

In [ ]:
import json, hashlib, random, shutil, time, gc, math
import torch
from PIL import Image
from datasets import load_dataset, concatenate_datasets

SEED = 42
DATA_DIR = "/kaggle/working/data"; IMG_DIR = f"{DATA_DIR}/images"
os.makedirs(IMG_DIR, exist_ok=True)

ds = load_dataset("flaviagiammarino/vqa-rad")
all_ds = concatenate_datasets([ds[k] for k in ds.keys()])
print(all_ds)
assert {"image", "question", "answer"} <= set(all_ds.column_names), all_ds.column_names

rows, seen = [], set()
for i, ex in enumerate(all_ds):
    img = ex["image"].convert("RGB")
    h = hashlib.md5(img.tobytes()).hexdigest()[:12]
    if h not in seen:
        img.save(f"{IMG_DIR}/{h}.jpg", quality=95); seen.add(h)
    ans = str(ex["answer"]).strip().lower()
    closed = ans in ("yes", "no")
    rows.append({"id": f"vqarad_{i}", "image_id": h, "image_path": f"images/{h}.jpg",
                 "question": str(ex["question"]).strip(), "answer": ans,
                 "question_type": "closed" if closed else "open",
                 "category": "closed_no" if ans == "no" else "normal"})

# split BY IMAGE so no image appears in two splits
ids = sorted(seen); random.Random(SEED).shuffle(ids)
n = len(ids); test_ids = set(ids[:int(.15*n)]); val_ids = set(ids[int(.15*n):int(.25*n)])
splits = {"train": [], "val": [], "test": []}
for r in rows:
    splits["test" if r["image_id"] in test_ids else "val" if r["image_id"] in val_ids else "train"].append(r)
for a in splits:
    for b in splits:
        if a < b:
            assert not ({r["image_id"] for r in splits[a]} & {r["image_id"] for r in splits[b]}), "image leakage"
for k, v in splits.items():
    json.dump(v, open(f"{DATA_DIR}/{k}.json", "w"), indent=1)
    print(k, len(v), "questions |", len({r['image_id'] for r in v}), "images |",
          sum(r['category']=='closed_no' for r in v), "closed_no")

In [ ]:
def upload(folder, path_in_repo):
    if not wtoken:
        print("No HF_WRITE_TOKEN secret: skipping upload. Download from the Output panel instead."); return
    from huggingface_hub import HfApi
    api = HfApi(token=wtoken); user = api.whoami()["name"]; repo = f"{user}/clinical-vlm-medgemma-lora"
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_folder(folder_path=folder, path_in_repo=path_in_repo, repo_id=repo)
    print("uploaded", folder, "->", repo, "/", path_in_repo)
upload(DATA_DIR, "data")

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/medgemma-4b-it"
PROMPT = "Answer the clinical question about this image briefly. {q}"
TIME_BUDGET_MIN = 150      # total training time budget
EFFECTIVE_BATCH = 16
LR = 1e-4
ADAPTER_DIR = "/kaggle/working/adapter"; CKPT_DIR = "/kaggle/working/adapter_ckpt"

# balance yes/no in the training set (false-yes bias is a known hallucination mode)
train = list(splits["train"])
yes = [r for r in train if r["answer"] == "yes"]; no = [r for r in train if r["answer"] == "no"]
rest = [r for r in train if r["answer"] not in ("yes", "no")]
if no and len(no) < len(yes):
    no = no + random.Random(SEED).choices(no, k=len(yes) - len(no))
train = yes + no + rest
random.Random(SEED).shuffle(train)
print("train samples after balancing:", len(train), "| yes", len(yes), "| no", len(no), "| open", len(rest))

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"
marker_ids = processor.tokenizer("<start_of_turn>model\n", add_special_tokens=False).input_ids

dt = torch.float32
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dt)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, quantization_config=bnb,
                                                    device_map={"": 0}, torch_dtype=dt)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj)"))
model.print_trainable_parameters()

def make_batch(samples):
    texts, images = [], []
    for s in samples:
        img = Image.open(f"{DATA_DIR}/{s['image_path']}").convert("RGB")
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT.format(q=s["question"])}]},
                {"role": "assistant", "content": [{"type": "text", "text": s["answer"]}]}]
        texts.append(processor.apply_chat_template(msgs, tokenize=False))
        images.append([img])
    b = processor(text=texts, images=images, padding=True, return_tensors="pt", add_special_tokens=False)
    labels = b["input_ids"].clone(); L = len(marker_ids)
    for i in range(labels.size(0)):
        ids = b["input_ids"][i].tolist(); pos = -1
        for j in range(len(ids) - L, -1, -1):
            if ids[j:j+L] == marker_ids: pos = j + L; break
        assert pos > 0, "answer marker not found - tokenization mismatch"
        labels[i, :pos] = -100
    labels[b["attention_mask"] == 0] = -100
    b["labels"] = labels
    return {k: v.to(0) for k, v in b.items()}

b = make_batch(train[:2])
print("sanity: tokens", b["input_ids"].shape, "| supervised tokens per sample", (b["labels"] != -100).sum(1).tolist())
print("supervised text:", processor.tokenizer.decode(b["labels"][0][b["labels"][0] != -100]))

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)

# autotune micro-batch size: largest that fits, then measure speed
BS, sec_per_sample = None, None
for bs in (4, 2, 1):
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        model.train(); bb = make_batch(train[:bs]); t0 = time.time()
        model(**bb).loss.backward(); torch.cuda.synchronize()
        sec_per_sample = (time.time() - t0) / bs
        opt.zero_grad(); BS = bs
        print(f"micro-batch {bs} OK: {sec_per_sample:.1f} s/sample, peak {torch.cuda.max_memory_allocated()/1024**3:.1f} GB"); break
    except torch.cuda.OutOfMemoryError:
        opt.zero_grad(); gc.collect(); torch.cuda.empty_cache(); print(f"micro-batch {bs}: out of memory")
assert BS, "even batch size 1 does not fit"
ACCUM = max(1, EFFECTIVE_BATCH // BS)
sec_per_step = sec_per_sample * BS * ACCUM
total_steps = max(5, int(TIME_BUDGET_MIN * 60 / (sec_per_sample * BS * ACCUM * 1.1)))
max_by_data = len(train) // (BS * ACCUM)
total_steps = min(total_steps, max_by_data * 2)   # at most 2 epochs
print(f"BS={BS} ACCUM={ACCUM} -> ~{sec_per_step:.0f}s per optimizer step | planned steps={total_steps} "
      f"(~{total_steps*BS*ACCUM} samples = {total_steps*BS*ACCUM/len(train):.2f} epochs)")

In [ ]:
def lr_at(step):
    warm = max(1, int(.05 * total_steps))
    if step < warm: return LR * (step + 1) / warm
    return LR * max(0.1, 1 - (step - warm) / max(1, total_steps - warm))

model.train(); t_start = time.time(); idx = 0; run_loss = []
for step in range(total_steps):
    for g in opt.param_groups: g["lr"] = lr_at(step)
    for _ in range(ACCUM):
        if idx + BS > len(train): idx = 0; random.Random(SEED + step).shuffle(train)
        loss = model(**make_batch(train[idx:idx+BS])).loss
        (loss / ACCUM).backward(); run_loss.append(float(loss)); idx += BS
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    opt.step(); opt.zero_grad()
    if (step + 1) % 5 == 0:
        el = (time.time() - t_start) / 60
        print(f"step {step+1}/{total_steps} | loss {sum(run_loss[-5*ACCUM:])/len(run_loss[-5*ACCUM:]):.3f} | {el:.0f} min elapsed")
    if (step + 1) % 25 == 0:
        model.save_pretrained(CKPT_DIR)
    if (time.time() - t_start) / 60 > TIME_BUDGET_MIN * 1.15:
        print("time budget exceeded, stopping early"); break
model.save_pretrained(ADAPTER_DIR)
json.dump({"base_model": MODEL_ID, "prompt_template": PROMPT, "lora_r": 16, "lora_alpha": 32,
           "train_compute_dtype": "float32", "steps_done": step + 1, "micro_batch": BS, "accum": ACCUM,
           "train_samples_seen": (step + 1) * BS * ACCUM, "minutes": round((time.time() - t_start) / 60, 1)},
          open(f"{ADAPTER_DIR}/train_info.json", "w"), indent=1)
print("saved adapter to", ADAPTER_DIR, "| final avg loss", sum(run_loss[-ACCUM*5:])/len(run_loss[-ACCUM*5:]))

In [ ]:
# quick sanity: 8 held-out validation questions with the trained adapter
model.eval(); model.config.use_cache = True
val = [r for r in splits["val"] if r["question_type"] == "closed"][:8]
for r in val:
    img = Image.open(f"{DATA_DIR}/{r['image_path']}").convert("RGB")
    msgs = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT.format(q=r["question"])}]}]
    inp = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(0)
    with torch.inference_mode():
        out = model.generate(**inp, max_new_tokens=16, do_sample=False)
    print(repr(r["question"][:70]), "| gold:", r["answer"], "| pred:", processor.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True).strip())

In [ ]:
upload(ADAPTER_DIR, "adapter")
print(open(f"{ADAPTER_DIR}/train_info.json").read())